# מפגש 22 — מעבדת תלמיד Audited
**משימת המקור:** bill_length_mm, bill_depth_mm, seed17, Test 25% ו־StratifiedKFold עם חמישה קיפולים. רשימת K הנתונה [1,3,5,15,51], macro-F1, השלמת ממוצע, StandardScaler ו־KNN בתוך Pipeline. **השלימו כל ??? לפני הרצה.** אין להשתמש ב־Test לבחירה; פתרון K אינו מוצג.

## 1. שלושת ציוני האימות הסינתטיים — חשבו תחילה ביד
הדגמת השיעור: 0.8,0.9,0.7. כתבו במילים מהו מכנה ממוצע הקיפולים ומה אינו כלול בו.

In [ ]:
import numpy as np
demo_scores=np.array([.8,.9,.7])
demo_mean=demo_scores.???()
print(demo_mean)

## 2. תשתית ונתונים — שימרו את היעד מחוץ למאפיינים

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df=pd.read_csv(url)
assert {"species","flipper_length_mm","body_mass_g","bill_length_mm","bill_depth_mm"}.issubset(df.columns)
assert df.shape[0]==344

In [ ]:
features=[???,???]
data=df.dropna(subset=["species"]).copy()
X=data[features].copy()
y=data["species"].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=???,random_state=???,stratify=y)
assert set(X_train.columns)==set(features)
assert len(X_train)+len(X_test)==len(y)

## 3. בדיקת חפיפה בין החלוקה החיצונית
מדוע חשוב שלא תהיה רשומה משותפת ל־Train ול־Test?

In [ ]:
assert len(X_train)==???
assert len(X_test)==???
assert set(X_train.index).isdisjoint(???)
assert X_train.shape[1]==???

## 4. חמשת הקיפולים
בעקבות המודרכת: השלימו את מספר הקיפולים והסבירו מה תפקיד `inner_validation`. אל תבצעו fit של Imputer/Scaler על X_train לפני הקריאה ל־cross_validate.

In [ ]:
cv=StratifiedKFold(n_splits=???,shuffle=True,random_state=42)
validation_counts=np.zeros(len(X_train),dtype=int)
for inner_train,inner_validation in cv.split(X_train,y_train):
    assert set(inner_train).isdisjoint(inner_validation)
    validation_counts[inner_validation]+=1
assert np.all(validation_counts==???)

## 5. מניעת זליגה
בעותק סינתטי נפרד יש ערכי אימון [0,2,4] וערך אימות [100]. מדוע ממוצע ההכנה חייב להילמד ב־Train בלבד?

In [ ]:
tiny_train=np.array([[0.],[2.],[4.]])
tiny_validation=np.array([[100.]])
safe_scaler=StandardScaler().fit(???)
unsafe_scaler=StandardScaler().fit(np.vstack([tiny_train,???]))
print(safe_scaler.mean_,unsafe_scaler.mean_)

## 6. מודלים מועמדים — אין לשנות את רשימת K מן המקור
השתמשו ב־scoring='f1_macro', חמשת הקיפולים ו־Pipeline. לפני הרצה: נסחו מי לומד ממוצע וסטיית תקן בכל קיפול.

In [ ]:
candidates=[1,3,5,15,51]
rows=[]
for k in candidates:
    pipe=make_pipeline(SimpleImputer(strategy=???),StandardScaler(),
                       KNeighborsClassifier(n_neighbors=???,weights="uniform",metric="euclidean"))
    folds=cross_validate(pipe,X_train,y_train,cv=???,scoring=???,return_train_score=True,n_jobs=1)
    rows.append({"K":k,"train_mean":folds["train_score"].mean(),
                 "cv_mean":folds["test_score"].mean(),
                 "cv_std":folds["test_score"].std(ddof=0)})
results_df=pd.DataFrame(rows)
display(results_df)

## 7. פירוש cv_results
בדקו שיש חמש שורות; `cv_std` הוא פיזור חמשת הציונים ולא רווח סמך. פער בין Train לאימות אינו הוכחה יחידה להתאמת יתר.

In [ ]:
assert len(results_df)==???
assert results_df["K"].tolist()==???
assert results_df["cv_mean"].between(???,???).all()
assert results_df["cv_std"].ge(???).all()

## 8. בחירה והערכה
בחרו לפי cv_mean הגבוה ביותר ובשוויון מדויק K הקטן; בנו Pipeline חדשה לכל Outer Train, ורק אז העריכו Test.

In [ ]:
ranked=results_df.sort_values(["cv_mean","K"],ascending=[False,True])
best_k=int(ranked.iloc[???]["K"])
final_model=make_pipeline(SimpleImputer(strategy="mean"),StandardScaler(),
                          KNeighborsClassifier(n_neighbors=???,weights="uniform",metric="euclidean"))
final_model.fit(???,???)
predictions=final_model.predict(???)
print("Outer Test macro-F1:",f1_score(y_test,predictions,average=???))
print("Outer Test Accuracy:",accuracy_score(y_test,predictions))

## 9. כרטיס ניסוי לפרויקט
תעדו שתי תכונות ויחידות, seed17, רשימת K, macro-F1, מספר קיפולים, כלל הכרעה, K נבחר, פלט Test לאחר בחירה ומגבלה. מתי יש להשתמש בחלוקה לפי קבוצות/זמן? מדוע changing random_state אינו מחזיר 'Test חדש'?